In this notebook, I classify the videos into categories based on their title.

Create the connection to the database.

In [1]:
import sqlite3
import pandas as pd 

conn = sqlite3.connect('../data/lafc_content.db')

Open the stored SQL query with combined video vs lafc match context table, and pull it into the base_df dataframe.

In [2]:
with open('../sql/videos_vs_lafc_match_context.sql') as f:
    query = f.read()
base_df = pd.read_sql(query, conn)

base_df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,season,kickoff_utc,...,home_away,goals_for,goals_against,lafc_points,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match
0,J8KtvKKDsBI,Inside LAFC | Episode 211 - A Strong Start,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T08:27:48Z,PT49M46S,1129,88,4,2026,2026-07-18T02:25:00Z,...,A,3,0,24,15,7,20,15,5,3.25
1,6IGiJLX6zIA,Sonny's goal from pitchside 🤳,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T05:15:15Z,PT17S,9169,901,20,2026,2026-07-18T02:25:00Z,...,A,3,0,24,15,7,20,15,5,3.12
2,dk5FTY2zHEI,Son Heung-Min | EVERY ANGLE of his derby goal ...,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-20T07:17:45Z,PT2M10S,10782,1373,100,2026,2026-07-18T02:25:00Z,...,A,3,0,24,15,7,20,15,5,2.20
3,WGLkpecuCyA,LAFC Weekly | Episode 15 | 2026,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-19T01:00:21Z,PT21M,2106,167,9,2026,2026-07-18T02:25:00Z,...,A,3,0,24,15,7,20,15,5,0.94
4,rjdbdSE3TNo,A Night To Remember | LAG vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-18T22:58:28Z,PT25S,2759,457,31,2026,2026-07-18T02:25:00Z,...,A,3,0,24,15,7,20,15,5,0.86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3478,EZP7m0-qpGA,"""We can't wait."" | Vela On Banc of California ...",,2018-03-06T20:17:55Z,PT1M1S,12628,170,10,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,1.93
3479,SME_lvoBvuw,WATCH: A closer look at the first goal in LAFC...,,2018-03-06T20:17:53Z,PT30S,1239,28,1,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,1.93
3480,aAeIq6bAdTI,All-Access: Behind the Scenes of LAFC's First ...,"An exclusive, behind the scenes look at the fi...",2018-03-05T20:14:02Z,PT3M6S,6600,194,14,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,0.93
3481,ff8VZHNeD9k,Diego Rossi Scores The First Goal in LAFC Hist...,Diego Rossi scored a beautiful goal in LAFC's ...,2018-03-05T02:52:36Z,PT1M9S,13899,250,18,2018,2018-03-04T22:00:00Z,...,A,1,0,0,0,0,0,0,0,0.20


Import Tfid and kMeans from scikit learn.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

Create a series of titles from the base data frame, I'm ignoring the description because it's repeated across different types of videos, and biases clustering.

In [4]:
text = base_df['title'].fillna('')
text.head()

0           Inside LAFC | Episode 211 - A Strong Start
1                        Sonny's goal from pitchside 🤳
2    Son Heung-Min | EVERY ANGLE of his derby goal ...
3                      LAFC Weekly | Episode 15 | 2026
4                    A Night To Remember | LAG vs LAFC
Name: title, dtype: str

Adding (too) commonly occuring title words to the English stop words list:

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
custom = ENGLISH_STOP_WORDS.union({'lafc','https','com','www','la','los','angeles','ep','episode'})

Vectorize the text series, into a sparse matrix, using tfid, and load it in a feature variable X.

In [6]:
vec = TfidfVectorizer(
    stop_words=list(custom),   # use my custom words as stop_words
    ngram_range=(1, 2),     # single words AND 2-word phrases ("inside lafc")
    min_df=5,               # ignore terms that appear in fewer than 5 videos (rare noise)
    max_df=0.4,
    max_features=500,       # cap vocabulary size — keeps it fast/focused
)
X = vec.fit_transform(text)

Create the kmeans model and assign each video a cluster.

In [7]:
k = 8
km = KMeans(n_clusters=k, random_state=11, n_init=10)
base_df['cluster'] = km.fit_predict(X)

Examining the clusters:

In [8]:
terms = vec.get_feature_names_out()
for c in range(k):
    top_idx = km.cluster_centers_[c].argsort()[-10:][::-1]   # 10 highest-weight terms
    keywords = ", ".join(terms[i] for i in top_idx)
    examples = base_df[base_df.cluster == c]['title'].head(3).tolist()
    print(f"\ncluster {c} (n={(base_df.cluster==c).sum()})")
    print(f"  keywords: {keywords}")
    print(f"  examples: {examples}")


cluster 0 (n=2042)
  keywords: goal, cherundolo, weekly, vela, carlos, carlos vela, bouanga, crest, season, presented
  examples: ["Sonny's goal from pitchside 🤳", 'LAFC Weekly | Episode 15 | 2026', 'Sonny back on the scoresheet 🫡']

cluster 1 (n=46)
  keywords: real, real salt, salt lake, lake, salt, vs real, vs, highlights, official, highlights vs
  examples: ['Real España vs LAFC | All 6 Goals', "LAFC+ | Ep. 81 - It's Getting Real", 'LAFC vs. Real Salt Lake | MATCH HIGHLIGHTS | BOUANGA HAT TRICK']

cluster 2 (n=629)
  keywords: vs, highlights, media, highlights vs, match, goal, prematch, prematch media, postmatch, postmatch media
  examples: ['Son Heung-Min | EVERY ANGLE of his derby goal | LAG vs LAFC', 'A Night To Remember | LAG vs LAFC', 'LAG vs LAFC | Postmatch Media']

cluster 3 (n=212)
  keywords: gold, black gold, black, gold insider, insider, gold weekly, weekly, presented, goal, ryan
  examples: ['LA is Black & Gold.', 'Black & Gold Insider Ep. 55 | Yevhen Cheberko', 'Yev

Based on the clustered keywords, I created a rules based classify function. I also created a format family dictionary, to group related cateogries, post function.

In [9]:
def classify_format(title):
    """
    Rule-based format classifier for LAFC videos.
    Order matters: named series first, then press/interview, then match content,
    then themed content, catch-all last. First match wins. Title-only, no regex.
    """
    t = str(title).lower()

    # 1. Named recurring series
    if 'inside lafc' in t:                                          return 'inside_lafc'
    if 'gold insider' in t:                                         return 'black_and_gold'   # the SHOW only
    if 'is black & gold' in t or 'is black and gold' in t:          return 'signing'          # "X is Black & Gold"
    if 'mvp podcast' in t:                                          return 'mvp_podcast'
    if 'acción' in t or 'accion' in t:                             return 'accion_lafc'
    if 'lafc weekly' in t or 'weekly | ' in t:                     return 'lafc_weekly'
    if 'lafc+' in t or 'lafc +' in t:                              return 'lafc_plus'
    if 'on the mic' in t:                                           return 'on_the_mic'
    if 'behind the crest' in t:                                     return 'behind_the_crest'
    if 'away days' in t:                                            return 'away_days'
    if 'a lot more to prove' in t:                                  return 'more_to_prove'
    if 'staying home' in t:                                         return 'staying_home'
    if '안녕 lafc' in t:                                            return 'korean_series'
    if 'podcast' in t:                                              return 'other_podcast'

    # 2. Press / interview (keyword-based)
    if 'postmatch' in t or 'post-match' in t:                      return 'postmatch_media'
    if 'prematch' in t or 'pre-match' in t:                        return 'prematch_media'
    if 'conference' in t or 'presser' in t or 'media availab' in t:  # 'availab' catches the typo
        return 'presser'
    if any(w in t for w in ['in touch','speaks','discusses','talks ',
                            'thoughts','reacts','reaction','breaks down','sits down']):
        return 'interview'

    # 3. Match content
    if 'highlight' in t or 'all goals' in t or 'every goal' in t or 'goal scored' in t \
       or 'top ten goals' in t or 'top 10 goals' in t or 'best goals' in t or 'top goals' in t:
        return 'highlights'
    if ('goal:' in t or t.startswith('goal ') or t.startswith('goal!')
            or '| goal' in t or 'every angle' in t or 'wondergoal' in t or 'golazo' in t
            or 'game winner' in t or 'from the spot' in t or 'from pitchside' in t):
        return 'goal_clip'
    if 'save of the match' in t or 'huge save' in t or 'leaping save' in t or 'big save' in t:
        return 'save_clip'
    if 'match frames' in t:                                         return 'match_frames'
    if 'preview' in t or 'keys to the match' in t:                 return 'match_preview'
    if 'recap' in t:                                                return 'recap'

    # 4. Themed / behind-the-scenes / features
    if 'behind the scenes' in t or 'sounds of' in t:               return 'behind_the_scenes'
    if 'training' in t or "mic'd up" in t or 'micd up' in t:       return 'training'
    if 'watch party' in t or 'watch along' in t or 'watch-along' in t or '110 football' in t:
        return 'watch_party'
    if 'built for it' in t:                                         return 'built_for_it'
    if any(w in t for w in ['get to know','player profile','lafc profile','join the club',
                            'on this day','cali to cali']):         return 'feature'

    # 5. Catch-all
    return 'unclassified'


FORMAT_FAMILY = {
    # produced recurring series
    'inside_lafc':      'show',
    'black_and_gold':   'show',
    'mvp_podcast':      'show',
    'accion_lafc':      'show',
    'lafc_weekly':      'show',
    'lafc_plus':        'show',
    'on_the_mic':       'show',
    'behind_the_crest': 'show',
    'away_days':        'show',
    'more_to_prove':    'show',
    'staying_home':     'show',
    'korean_series':    'show',
    'other_podcast':    'show',

    # match content
    'highlights':       'match',
    'goal_clip':        'match',
    'save_clip':        'match',
    'match_frames':     'match',
    'match_preview':    'match',
    'recap':            'match',

    # press / interview
    'postmatch_media':  'media',
    'prematch_media':   'media',
    'presser':          'media',
    'interview':        'media',

    # themed / features
    'behind_the_scenes':'behind_scenes',
    'training':         'behind_scenes',
    'watch_party':      'watch_party',
    'built_for_it':     'feature',
    'feature':          'feature',
    'signing':          'signing',

    # catch-all
    'unclassified':     'unclassified', 
}

Apply the classify function, and map the format family, to two new columns in the base_df, then count them:

In [10]:
base_df['content_type']   = base_df['title'].apply(classify_format)
base_df['format_family']  = base_df['content_type'].map(FORMAT_FAMILY)

base_df['content_type'].value_counts()

content_type
unclassified         1830
highlights            243
goal_clip             169
inside_lafc           164
lafc_weekly           111
lafc_plus             103
accion_lafc            86
mvp_podcast            82
recap                  65
match_preview          64
signing                62
prematch_media         61
postmatch_media        57
black_and_gold         55
interview              53
behind_the_crest       49
feature                47
watch_party            39
training               36
presser                34
behind_the_scenes      21
korean_series          12
match_frames           11
away_days               9
on_the_mic              8
save_clip               4
staying_home            3
other_podcast           2
built_for_it            2
more_to_prove           1
Name: count, dtype: int64

Even with all my rules, many of the videos are not easily classifiable by title, those are assigned the category of "unclassified".

In [11]:
base_df[['title', 'format_family']].sample(30)

,title,format_family
928,Inside LAFC | Ep. 152 - Six goals in Portland,show
1999,Black & Gold Weekly | A Title On The Line,show
2165,In Touch with Steve Cherundolo Ep. 19 | Gettin...,media
1781,Olivier Giroud to join LAFC! #lafc #football ...,unclassified
1215,Celebrating 10 Years of LAFC | Thank You,unclassified
2613,Get To Know | Bryce Duke,feature
2927,Highlights | LAFC vs Montreal Impact,match
2588,LAFC Launches The Black & Gold Community Relie...,unclassified
183,Tyler's first goal in Black & Gold | LAFC vs ORL,unclassified
598,Son Heung-Min's thoughts after his debut.,media


I tuned the classification based on rules as best I could, but I still had ≈ 1800 videos that I was labeling "unclassified" they were mostly short social clips without a standard title, but some were interview clips, so I decided to try a light machine learning model on the "unclassified" set to see if I could label those more finely. 

In [12]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/laptop_02/Documents/data_projects/lafc_content/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9383.23it/s]


In [13]:
CATEGORIES = {
    'goal_clip':   'a video clip of a single goal being scored in a match',
    'highlights':  'match highlights or a compilation of multiple goals',
    'interview':   'a player or coach speaking, quote, press interview or reaction',
    'feature':     'a player profile, personal story, or getting-to-know feature',
    'behind_the_scenes': 'behind the scenes footage, training, or documentary content',
    'match_preview':'a preview or build-up before an upcoming match',
    'recap':       'a season or match recap looking back',
    'misc_social_clip': 'a short hype or social media clip, slogan, or promotional post',
}

In [14]:
cat_names = list(CATEGORIES.keys())
cat_embeddings = model.encode(list(CATEGORIES.values()))

In [15]:
unclassified = base_df[base_df['content_type'] == 'unclassified'].copy()
title_embeddings = model.encode(unclassified['title'].fillna('').tolist(), show_progress_bar=True)

Batches: 100%|██████████| 58/58 [00:00<00:00, 59.94it/s]


In [16]:
import numpy as np
sims = util.cos_sim(title_embeddings, cat_embeddings).numpy()   # shape: (n_titles, n_categories)
best_idx   = sims.argmax(axis=1)     # which category scored highest per title
best_score = sims.max(axis=1)        # how strong that match was
unclassified['ml_label'] = [cat_names[i] for i in best_idx]
unclassified['ml_score'] = best_score

In [17]:
THRESHOLD = 0.30   # tune this — see below
unclassified['ml_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'          # weak match -> genuine miscellany
)

In [18]:
unclassified[['title', 'ml_label', 'ml_score']].sample(30)

,title,ml_label,ml_score
124,Bogusz Banger From The Archives ⏮️,misc_social_clip,0.166532
1049,Eager to get started.,misc_social_clip,0.248271
2627,LAFC Players On Carlos Vela's Leadership,misc_social_clip,0.292473
2237,LAFC & KCOP 13 Team Up For All 2022 English-La...,misc_social_clip,0.285180
2703,LAFC Goal Rush Inaugural Season | Top 5 Goals ...,highlights,0.366889
723,First in the Black & Gold for Javairô Dilrosun 👊,misc_social_clip,0.173977
1488,"Cherundolo: Right Now, Our Difference-Makers A...",misc_social_clip,0.181257
1677,Off the Pitch with Ilie Sánchez,goal_clip,0.358610
298,Ready to fight for the Club 🫡,misc_social_clip,0.203376
811,Vintage Lloris 🧤,misc_social_clip,0.161813


In [19]:
unclassified['ml_score'].describe()

count    1830.000000
mean        0.249290
std         0.093928
min         0.024242
25%         0.182427
50%         0.246149
75%         0.309137
max         0.584862
Name: ml_score, dtype: float64

In [20]:
unclassified['dur_min'] = pd.to_timedelta(unclassified['duration']).dt.total_seconds() / 60

u = unclassified.sort_values('ml_score', ascending=False)

print("=== HIGH scores (top matches) ===")
print(u.head(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== MID scores (around the median) ===")
print(u.iloc[900:915][['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== LOW scores (weakest) ===")
print(u.tail(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

=== HIGH scores (top matches) ===
                                                                         title    ml_label  ml_score  dur_min
3479                    WATCH: A closer look at the first goal in LAFC history   goal_clip     0.585      0.5
405                                             ✌️ Regular season matches left       recap     0.570      0.3
714                                    Goals in back-to-back matches for Ordaz  highlights     0.527      0.5
853                                                     4️⃣ goals in one place  highlights     0.526      0.6
2882                       11 Goals Over In Our Last 2 Games | Watch Them All!  highlights     0.524      1.0
1446                    1 Goal = 3 Celebrations | #LAFC #MLS #LeaguesCup #Goal  highlights     0.518      0.2
2917                Anatomy Of A Goal | Diego Rossi vs Portland Timbers 6/1/19   goal_clip     0.517      2.4
3106                                            The Can't Miss Matches Of 2019       r

The light ml model did well at recognizing goal and highlight clips, but it was missing interview clips. I decide to provide the model with category vectors based on actual examples titles for each category, instead of providing it with text descriptions of the categories, to see if that would improve its classification.

In [21]:
PROTOTYPES = {
    'goal_clip': [
        # structured (with score line)
        "GOAL: M. Bogusz vs VAN, 1'",
        "Denis Bouanga breaks the tie! LAFC 2 - 1 HOU",
        "Diego Rossi opens the scoring, LAFC 1 - 0 Dallas",
        # short descriptive goal moments (no score line — the leaking shape)
        "Sonny picks his spot 🎯",
        "Near post finish by Bouanga 😮‍💨",
        "Bouanga chips the keeper | ALL ANGLES",
        "SONNY FROM DISTANCE 🚀",
        "Timmy's strike from the top of the box",
    ],

    'highlights': [
        "Highlights | LAFC vs FC Dallas",
        "Full Highlights | 3-0 | LAFC vs. Colorado Rapids",
        "MATCH HIGHLIGHTS | LAFC vs Seattle Sounders",
        "Every Goal From LAFC's Inaugural MLS Season",
        "11 Goals Over In Our Last 2 Games | Watch Them All",
    ],
    'interview': [
        "Cherundolo: We'll Need Effort Again Against Austin",
        "Ebobisse: Feeling more and more confident by the day",
        "Bradley Addresses Media After Mark-Anthony Kaye Trade",
        "Bouanga speaks on his hat trick",
        "Nguyen: This Is Where You Start To Play For Playoff Positions",
        "State Of The Union | Tom Penn",
        "Hollingshead: Huge Result For Us, Three Points On The Road",
    ],
    'feature': [
        "Get To Know Kwadwo Opoku",
        "LAFC Profile | From Norway to LA, Adama Diomande",
        "Building A Legacy | Carlos Vela's Past & Future With LAFC",
        "Join The Club | Juan Pinto",
        "The Call-Up | Christian Ramirez",
    ],
    'behind_the_scenes': [
        "Behind The Scenes | 2026 Primary Kit Shoot",
        "Sounds of Training | First Week Back",
        "A Look Behind The Scenes With Equipment Manager Scott Tranilla",
        "Inside the locker room after the win",
    ],
    'match_preview': [
        "LAFC at LA Galaxy - Match Preview",
        "Keys To The Match | LAFC vs Seattle",
        "Previewing the road trip to Colorado",
        "What to watch for ahead of LAFC vs Austin FC",
    ],
    'recap': [
        "2023 LAFC Season Recap",
        "Recap | LAFC vs Colorado Rapids",
        "Looking Back At The 2022 MLS Cup Run",
        "Year In Review | 2021 Season",
    ],
    'misc_social_clip': [
        "24 Hours ⏳",
        "The dagger 🗡️",
        "99 is electric ⚡️",
        "Ready to fight for the Club 🫡",
        "Pressure is a privilege",
        "It's about the collective.",
    ],
}

In [22]:
import numpy as np

cat_names = list(PROTOTYPES.keys())
cat_vectors = []
for name in cat_names:
    ex_embs = model.encode(PROTOTYPES[name])   # embed that category's example titles
    cat_vectors.append(ex_embs.mean(axis=0))   # average -> one "centroid" per category
cat_vectors = np.vstack(cat_vectors)

In [23]:
from sentence_transformers import util

sims = util.cos_sim(title_embeddings, cat_vectors).numpy()
best_idx = sims.argmax(axis=1)
unclassified['ml_label'] = [cat_names[i] for i in best_idx]
unclassified['ml_score'] = sims.max(axis=1)

In [24]:
THRESHOLD = 0.47   # tuned based on running the model a few times at looking at resulting ml scores against titles
unclassified['ml_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'
)

In [25]:
unclassified[['title', 'ml_label', 'ml_score']].sample(30)

,title,ml_label,ml_score
3058,Carlos Vela Named MLS Player Of The Week,misc_social_clip,0.466446
2531,Los Sueños Se Hacen Realidad | Andy Najar's Jo...,feature,0.497525
3325,World Cup Hero in Los Angeles: Heung-Min Son C...,misc_social_clip,0.325615
1476,Déjà Vu. Hugo Lloris ➡️ Denis Bouanga | #LAFC ...,goal_clip,0.519048
1649,Hugo et Denis Find Out They're 2024 MLS All-Stars,recap,0.521070
2122,Carlos Vela Strikes From Distance Against Minn...,goal_clip,0.472724
3474,Diego Rossi Scores First Goal In LAFC History,misc_social_clip,0.456084
2377,Faces Of LAFC | Oogie Lee: A First Generation ...,feature,0.475517
1485,Bouanga: The Best Performance Of Tonight Is Th...,goal_clip,0.610650
166,HUGOOOO 🧤,misc_social_clip,0.439329


In [26]:
unclassified['ml_score'].describe()

count    1830.000000
mean        0.444192
std         0.110852
min         0.122112
25%         0.368982
50%         0.446596
75%         0.516055
max         0.823427
Name: ml_score, dtype: float64

In [27]:
unclassified['dur_min'] = pd.to_timedelta(unclassified['duration']).dt.total_seconds() / 60

u = unclassified.sort_values('ml_score', ascending=False)

print("=== HIGH scores (top matches) ===")
print(u.head(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== MID scores (around the median) ===")
print(u.iloc[900:915][['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

print("\n=== LOW scores (weakest) ===")
print(u.tail(15)[['title','ml_label','ml_score','dur_min']].round({'ml_score':3,'dur_min':1}).to_string())

=== HIGH scores (top matches) ===
                                                                                       title       ml_label  ml_score  dur_min
310                                                    A Year In Review | LAFC's 2025 Season          recap     0.823      4.6
3152                                             MEMORABLE MATCHDAY | LAFC Makes MLS History     highlights     0.768      3.9
2594                                               LAFC In 30 | LAFC vs. FC Dallas - 5/19/19     highlights     0.737     30.0
3454                                     WATCH: All 3 Goals in LAFC's 3-4 Loss vs. LA Galaxy     highlights     0.736      2.2
3442                                 WHAT A MATCH: LAFC vs. Montreal Impact | April 21, 2018     highlights     0.735      4.0
3479                                  WATCH: A closer look at the first goal in LAFC history     highlights     0.733      0.5
1716                                                       2024 Preseason: LA

It did a better job at classifying, and I went back and adjusted the ml_score threshold (below that the model would default a row to misc_social_clip). Then I merged the data frame of previously unclassified, back into the base_df, adding a content_type_final_column.

In [32]:
import numpy as np

# 1. Apply the threshold -> final label for the unclassified rows
THRESHOLD = 0.47
unclassified['final_label'] = np.where(
    unclassified['ml_score'] >= THRESHOLD,
    unclassified['ml_label'],
    'misc_social_clip'
)

# FORMAT_FAMILY was written before the ML pass existed, so it has no entry
# for the one category only the model can produce. Add it here rather than
# editing the original dict, to keep the order of discovery visible.
FORMAT_FAMILY['misc_social_clip'] = 'social'

# 2. Start the merged column as the RULE labels (the trustworthy core)
base_df['content_type_final'] = base_df['content_type']

# 3. Overwrite ONLY the previously-unclassified rows with the ML result.
#    Aligns by index — works because `unclassified` kept base_df's index.
base_df.loc[unclassified.index, 'content_type_final'] = unclassified['final_label']

# 4. Re-map the coarse family on the merged labels
base_df['format_family'] = base_df['content_type_final'].map(FORMAT_FAMILY)

# 5. Sanity checks
print(base_df['content_type_final'].value_counts(), "\n")
print("misc_social_clip:", f"{(base_df.content_type_final=='misc_social_clip').mean():.0%}")
print("unmapped families (must be 0):", base_df['format_family'].isna().sum())

content_type_final
misc_social_clip     1136
highlights            448
goal_clip             365
inside_lafc           164
recap                 154
match_preview         141
feature               118
lafc_weekly           111
interview             109
lafc_plus             103
accion_lafc            86
mvp_podcast            82
signing                62
prematch_media         61
postmatch_media        57
black_and_gold         55
behind_the_crest       49
watch_party            39
training               36
presser                34
behind_the_scenes      21
korean_series          12
match_frames           11
away_days               9
on_the_mic              8
save_clip               4
staying_home            3
other_podcast           2
built_for_it            2
more_to_prove           1
Name: count, dtype: int64 

misc_social_clip: 33%
unmapped families (must be 0): 0
format_family
social           1136
match            1123
show              685
media             261
feature         

In [33]:
print(base_df['format_family'].value_counts(), '\n')

format_family
social           1136
match            1123
show              685
media             261
feature           120
signing            62
behind_scenes      57
watch_party        39
Name: count, dtype: int64 

